# 2.5 Shape of Distribution

**Goal:** describe symmetry, tails, and unusual observations without assuming every dataset is normal.

**Prerequisites:** means, medians, variance, quantiles, and z-scores. **Runtime:** Python 3.10+, standard library only. Examples are synthetic; outputs are saved. Run cells in order.

## Contents
1. Symmetrical distribution
2. Positive skewness
3. Negative skewness
4. Kurtosis
5. Outliers
6. Normality
7. ML distribution analysis, practice, and solutions

## 1. Symmetrical Distribution

A distribution is symmetric about $c$ when its reflected values on either side have the same distribution. An exactly symmetric finite dataset has matching values at equal distances from $c$.

**Worked example:** -2,-1,0,1,2 are symmetric about 0, with mean and median 0. Symmetry does not imply normality: a uniform distribution and a symmetric two-peaked mixture can also be symmetric.

When a finite mean exists, a symmetric distribution has mean equal to its center, which is also a median. Some symmetric heavy-tailed distributions do not have a finite mean, so the mean-center statement requires that condition.

A symmetric empirical dataset has zero third central moment. Conversely, zero moment-based skewness does not establish full symmetry; the third moment is only one summary.

## 2. Positive Skewness

A positively skewed or right-skewed distribution has an extended right tail. Examples often include durations, spending, and nonnegative counts. “Right” describes the long tail, not where most values cluster.

**Worked example:** 1,2,2,3,12 have median 2 and mean 4. The high value pulls the mean above the median. That ordering is a useful intuition for many unimodal examples, not a universal definition or theorem for every distribution.

## 3. Negative Skewness

A negatively skewed or left-skewed distribution has an extended left tail. A score on an easy test may cluster near its upper bound with a few low values.

**Worked example:** -12,-3,-2,-2,-1 have median -2 and mean -4, reflecting the preceding example. The sign of raw values is unrelated to the sign of skewness: subtracting 100 from every right-skewed value does not reverse its skewness.

### A precisely defined skewness statistic

For a nonconstant empirical dataset, let $m_k=\frac1n\sum_i(x_i-\bar x)^k$. The unadjusted moment coefficient is
$$g_1=\frac{m_3}{m_2^{3/2}}.$$

Some software instead reports a bias-adjusted coefficient, such as $\sqrt{n(n-1)}\,g_1/(n-2)$ for $n>2$. Conventions and small-sample behavior matter; the code explicitly uses the unadjusted version. Moment-based summaries can be unstable with extremes and may not exist for some theoretical heavy-tailed distributions.

In [1]:
import math
import random
import statistics

def quantile(values, p):
    """Linear interpolation at position (n-1)*p (often called type 7)."""
    xs = sorted(values)
    if not xs or not 0 <= p <= 1:
        raise ValueError("Need nonempty values and p in [0, 1]")
    position = (len(xs)-1)*p
    low = math.floor(position)
    high = math.ceil(position)
    return xs[low] + (position-low)*(xs[high]-xs[low])

def shape_moments(values):
    if len(values) < 2:
        raise ValueError("Need at least two observations")
    mean = statistics.mean(values)
    m2 = statistics.mean((x-mean)**2 for x in values)
    if m2 == 0:
        raise ValueError("Shape moments need nonzero variance")
    m3 = statistics.mean((x-mean)**3 for x in values)
    m4 = statistics.mean((x-mean)**4 for x in values)
    return m3/m2**1.5, m4/m2**2

for name, xs in [("symmetric", [-2, -1, 0, 1, 2]),
                 ("right", [1, 2, 2, 3, 12]),
                 ("left", [-12, -3, -2, -2, -1])]:
    skew, kurt = shape_moments(xs)
    print(name, "mean:", statistics.mean(xs), "median:", statistics.median(xs), "g1:", round(skew, 4))
assert math.isclose(shape_moments([-2, -1, 0, 1, 2])[0], 0, abs_tol=1e-12)

symmetric mean: 0 median: 0 g1: 0.0
right mean: 4 median: 2 g1: 1.4093
left mean: -4 median: -2 g1: -1.4093


## 4. Kurtosis

Population kurtosis, when finite and variance is positive, is
$$\beta_2=\frac{\mathbb E[(X-\mu)^4]}{\sigma^4}.$$
The normal distribution has **Pearson kurtosis** 3. **Excess kurtosis** subtracts 3, so normal excess kurtosis is 0. Empirical moment kurtosis uses $m_4/m_2^2$; sample bias adjustments differ across tools.

Kurtosis reflects fourth-moment tail/extreme-value behavior relative to the variance. It is not adequately described as only “peakedness”: distributions can have similar central peaks but very different tails, and vice versa.

Traditional labels compare kurtosis to the normal reference: **leptokurtic** (excess > 0), **mesokurtic** (excess = 0), and **platykurtic** (excess < 0). Equal kurtosis to a normal does not prove normality, and high kurtosis does not alone identify a particular tail model.

**Worked example:** an empirical dataset with many zeros and rare values -10 and 10 has large kurtosis even though it is perfectly symmetric and its skewness is zero. Fourth powers give large deviations great influence. Some theoretical distributions have infinite fourth moments; a finite sample can still produce a finite but unstable empirical kurtosis.

In [2]:
symmetric_with_extremes = [-10] + [0]*98 + [10]
skew, pearson = shape_moments(symmetric_with_extremes)
print("Skewness:", skew)
print("Pearson kurtosis:", pearson)
print("Excess kurtosis:", pearson-3)
assert math.isclose(skew, 0, abs_tol=1e-12)
assert math.isclose(pearson, 50)

Skewness: 0.0
Pearson kurtosis: 50.0
Excess kurtosis: 47.0


## 5. Outliers

An outlier is an observation unusually distant from a relevant reference pattern. It may be a recording error, a rare valid event, a different subgroup, or a meaningful anomaly. Outlying status depends on context and the model/reference population.

Two common screens are:

- **IQR rule:** flag below $Q_1-1.5\operatorname{IQR}$ or above $Q_3+1.5\operatorname{IQR}$.
- **Z-score rule:** flag large $|z|$, often above 3, when a mean/SD reference is meaningful. It is not a universal probability guarantee; skewed or heavy-tailed data can naturally produce many such values.

**Worked example:** for 1 through 8 plus 30, the IQR fences are -3 and 13, so 30 is flagged. Yet its z-score computed using the same empirical mean and SD is below 3: the extreme itself inflates the reference SD. This is a simple example of **masking**.

Review units, data-entry mistakes, duplicates, collection changes, and subgroup structure before acting. Deleting valid extremes can bias an average, hide minority conditions, or remove the fraud cases being modeled. If capping, robust transformations, or exclusions are justified, fit thresholds on training data and document the decision.

A value can be ordinary marginally yet unusual jointly: a combination of age and experience might be implausible even if each alone is in range. Univariate fences do not solve multivariate anomaly detection.

In [3]:
xs = [1, 2, 3, 4, 5, 6, 7, 8, 30]
q1, q3 = quantile(xs, 0.25), quantile(xs, 0.75)
iqr = q3-q1
mean, sd = statistics.mean(xs), statistics.pstdev(xs)
for x in xs:
    fence_flag = x < q1-1.5*iqr or x > q3+1.5*iqr
    z = (x-mean)/sd
    if fence_flag or abs(z) > 3:
        print("Value:", x, "IQR flag:", fence_flag, "z-score:", round(z, 4))
assert 30 > q3+1.5*iqr
assert abs((30-mean)/sd) < 3

Value: 30 IQR flag: True z-score: 2.7309


## 6. Normality

A normal distribution is a particular continuous, symmetric, unimodal bell-shaped model:
$$f(x)=\frac1{\sigma\sqrt{2\pi}}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right),\qquad\sigma>0.$$
It has mean = median = mode = $\mu$, variance $\sigma^2$, skewness 0, and excess kurtosis 0. These summaries alone are not sufficient to establish normality.

Under an actual normal model, approximately 68.27%, 95.45%, and 99.73% lie within 1, 2, and 3 population SDs of the mean. These are model-specific probabilities, not universal facts for arbitrary data or exact counts in a finite sample.

Use a combination of histogram/density views, a **normal Q-Q plot** comparing observed quantiles to normal theoretical quantiles, and substantive knowledge. Approximate linearity in a Q-Q plot supports a normal-like shape; tail departures, skew, or multiple regimes can reveal mismatch. Formal normality tests can be underpowered with tiny samples and detect minor departures with huge samples. Failure to reject is not proof of normality.

**Worked example:** the code compares standardized sample order statistics to standard-normal quantiles at plotting positions $(i-0.5)/n$. A Q-Q plot would draw these pairs. A handful of matching central quantiles does not validate the tails; inspect the whole range.

In many regression settings, normality assumptions concern conditional errors/residuals for particular inferential procedures, not every input feature. Many ML algorithms do not require normal features. The central limit theorem concerns suitably normalized sums/means under conditions, not a guarantee that a large raw dataset becomes normal.

In [4]:
rng = random.Random(10)
normal_sample = sorted(rng.gauss(0, 1) for _ in range(200))
mean, sd = statistics.mean(normal_sample), statistics.pstdev(normal_sample)
standardized = [(x-mean)/sd for x in normal_sample]
reference = statistics.NormalDist()
print("Selected normal Q-Q pairs: theoretical, observed standardized")
for index in [0, 19, 99, 179, 199]:
    p = (index+0.5)/len(standardized)
    print(round(reference.inv_cdf(p), 3), round(standardized[index], 3))
for k in [1, 2, 3]:
    print(f"Normal-model probability within {k} SD:", reference.cdf(k)-reference.cdf(-k))

Selected normal Q-Q pairs: theoretical, observed standardized
-2.807 -2.5
-1.296 -1.218
-0.006 -0.022
1.267 1.235
2.807 2.858
Normal-model probability within 1 SD: 0.6826894921370859
Normal-model probability within 2 SD: 0.9544997361036416
Normal-model probability within 3 SD: 0.9973002039367398


## 7. AI/ML distribution analysis

For a positive skewed feature, a log transform may compress the high tail. `log1p(x)` can be useful for nonnegative values including zero, but it changes interpretation and does not guarantee normality. Do not add arbitrary constants to accommodate negative values without considering what the transform means.

Compare training, validation, and deployment distributions by feature and subgroup. Changes in skew, tail frequency, missingness, or label mix can expose collection drift or a changed population. A summary change alone cannot diagnose its cause, and equal marginal distributions do not guarantee equal joint relationships.

Scaling and transformations must be fit or chosen without peeking at held-out outcomes. Keep rare valid events if they matter to deployment rather than forcing the data to resemble an ideal bell curve.

In [5]:
amounts = [0, 1, 2, 3, 5, 10, 50, 200]
transformed = [math.log1p(x) for x in amounts]
print("Raw skewness:", shape_moments(amounts)[0])
print("Log1p skewness:", shape_moments(transformed)[0])
print("Original:", amounts)
print("Transformed:", [round(x, 3) for x in transformed])
print("A smaller sample skewness does not prove normality.")

Raw skewness: 2.040334765855136
Log1p skewness: 0.7613577979675671
Original: [0, 1, 2, 3, 5, 10, 50, 200]
Transformed: [0.0, 0.693, 1.099, 1.386, 1.792, 2.398, 3.932, 5.303]
A smaller sample skewness does not prove normality.


## Practice

1. Is a symmetric distribution necessarily normal? Give a counterexample.
2. Does positive skewness mean most observations are on the right side of a plot?
3. Convert Pearson kurtosis 5.4 to excess kurtosis. What is normal excess kurtosis?
4. Why can a large observation avoid a z>3 screen when mean and SD are fit on the same data?
5. Can a value beyond an IQR fence be valid and important for ML?
6. Does a large sample size make the raw observations normally distributed?
7. A dataset has mean=median and skewness close to zero. Is this enough to prove normality?
8. What does a normal Q-Q plot compare? Why inspect tails as well as the center?

## Solutions

1. No. A continuous uniform distribution or a symmetric bimodal mixture is not normal.
2. No. It describes an extended right tail; the mass can be concentrated at lower values.
3. $5.4-3=2.4$; normal excess kurtosis is 0.
4. It can inflate the fitted mean and especially SD, reducing its own standardized distance (masking).
5. Yes. Rare high costs, fraud events, or genuine extreme conditions may be essential deployment cases.
6. No. Under appropriate conditions the CLT concerns sample means/sums, not the shape of raw data.
7. No. Those are limited summaries; nonnormal distributions can share them.
8. Observed quantiles/order statistics versus theoretical normal quantiles. Many distributions resemble one another centrally but differ substantially in their tails.

**Readiness check:** distinguish symmetry from normality, Pearson from excess kurtosis, and an outlier flag from an error determination.

**Next:** [2.6 Data Visualization](6_Data-Visualization.ipynb).